# Round 3: Japanese GP Qualifying Analysis

## Imports and Data Loading

In [ ]:
import fastf1
import numpy as np
import matplotlib.pyplot as plt
from fastf1 import plotting
import plotly.graph_objects as go
from scipy.signal import savgol_filter
from src.plotset import setup_plot, save_fig

In [ ]:
fastf1.Cache.enable_cache('./f1_cache')
fastf1.Cache.get_cache_info()

In [ ]:
# ==========================================
# 1. Configuration
# ==========================================
EXAGGERATION_FACTOR = 20.0 
SMOOTHING_WINDOW = 101  # Increased for smoother look; must be odd
POLY_ORDER = 3         
TOTAL_FRAMES = 500     # Higher = smoother rotation
CAMERA_RADIUS = 3.0    # Zoom level

# # Load data
session_25 = fastf1.get_session(2025, 'Japan', 'Q')
session_26 = fastf1.get_session(2026, 'Japan', 'Q')
session_25.load()
session_26.load()

# ==========================================
# 2. Telemetry Processing
# ==========================================
lap_25 = session_25.laps.pick_fastest().get_telemetry().add_distance()
lap_26 = session_26.laps.pick_drivers('RUS').pick_fastest().get_telemetry().add_distance()

max_dist = max(lap_25['Distance'].max(), lap_26['Distance'].max())
ref_dist = np.linspace(0, max_dist, 2000) 

v25 = np.interp(ref_dist, lap_25['Distance'], lap_25['Speed'])
v26 = np.interp(ref_dist, lap_26['Distance'], lap_26['Speed'])
x_raw = np.interp(ref_dist, lap_26['Distance'], lap_26['X'])
y_raw = np.interp(ref_dist, lap_26['Distance'], lap_26['Y'])

# CENTER THE DATA: This makes the rotation "spin" around the track center
x = x_raw - np.mean(x_raw)
y = y_raw - np.mean(y_raw)

delta_speed = savgol_filter(v26 - v25, SMOOTHING_WINDOW, POLY_ORDER)
z_height = delta_speed * EXAGGERATION_FACTOR

In [ ]:
# ==========================================
# 3. Visualization
# ==========================================
fig = go.Figure()

# The Delta Curtain
fig.add_trace(go.Surface(
    x=np.array([x, x]), y=np.array([y, y]), z=np.array([np.zeros_like(z_height), z_height]),
    surfacecolor=np.array([delta_speed, delta_speed]),
    colorscale='RdYlGn', cmid=0,
    hovertemplate="Delta: %{surfacecolor:.1f} km/h<extra></extra>",
    colorbar=dict(title="km/h", thickness=20, x=0.9)
))

# Clean White Track Line
fig.add_trace(go.Scatter3d(
    x=x, y=y, z=np.zeros_like(x),
    mode='lines', line=dict(color='white', width=6),
    hoverinfo='none'
))

# ==========================================
# 4. Animation & Clean Layout
# ==========================================

# Create rotation frames
frames = []
for i in range(TOTAL_FRAMES):
    theta = (i / TOTAL_FRAMES) * (2 * np.pi)
    frames.append(go.Frame(
        layout=dict(scene=dict(camera=dict(
            eye=dict(
                x=CAMERA_RADIUS * np.cos(theta), 
                y=CAMERA_RADIUS * np.sin(theta), 
                z=1.2
            )
        ))),
        name=f"frame{i}"
    ))

fig.frames = frames

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor='black',  # Background of the entire canvas
    plot_bgcolor='black',   # Background of the plot area
    margin=dict(l=0, r=0, b=0, t=40),
    scene=dict(
        aspectmode='data',
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(
            visible=False, # Keeps the axes hidden as requested
            showgrid=False,
            showbackground=False,
        ),
        camera=dict(eye=dict(x=CAMERA_RADIUS, y=0, z=1.2)) # Initial position
    ),
    updatemenus=[dict(
        type="buttons",
        buttons=[
            dict(label="Play Rotation",
                 method="animate",
                 args=[None, {"frame": {"duration": 16, "redraw": True}, "fromcurrent": True, "mode": "immediate", "loop": True}]),
            dict(label="Pause",
                 method="animate",
                 args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}])
        ],
        x=0.1, y=0.1,
        font=dict(color="black") # Visible on white button background
    )],
    title=dict(text="Suzuka Speed Delta: 2026 vs 2025 (Animated)", font=dict(size=20))
)

fig.show(renderer="browser")

In [ ]:
corners = session_25.get_circuit_info().corners
rotation = session_25.get_circuit_info().rotation

In [ ]:
ref_lap = session_25.laps.pick_drivers('VER').pick_fastest().get_telemetry().copy()
comp_lap = session_26.laps.pick_drivers('RUS').pick_fastest().get_telemetry().copy()

In [ ]:
mult = ref_lap.Distance.iloc[-1]/comp_lap.Distance.iloc[-1]

ref_dist = ref_lap.Distance.to_numpy()
ref_time = ref_lap.Time.dt.total_seconds().to_numpy()
comp_dist = (comp_lap.Distance * mult).to_numpy()
comp_time = comp_lap.Time.dt.total_seconds().to_numpy()

In [ ]:
comp_time = np.interp(ref_dist, comp_dist, comp_time)

In [ ]:
setup_plot(axeslabel=24, figtitle=28, legendfont=22)

ant_color = '#FFFFFF'#plotting.get_driver_color(session=session_25, identifier='VER')
ham_color = plotting.get_driver_color(session=session_26, identifier='RUS')

fig, ax = plt.subplots(2,1,figsize=(15,8),height_ratios=[0.7,0.3],sharex=True)
ax[0].plot(ref_lap['Distance'],ref_lap['Speed'],color=ant_color,linewidth=3, label='2025 - VER')
ax[0].plot(comp_lap['Distance'],comp_lap['Speed'],color=ham_color,linewidth=3, label='2026 - RUS')

ax[0].vlines(x=corners['Distance'], ymin=0, ymax=340,
          linestyles='dotted', colors='grey')
for _, corner in corners.iterrows():
    txt = f"{corner['Number']}{corner['Letter']}"
    ax[0].text(corner['Distance'], 340, txt,
            va='center_baseline', ha='center', size=15, color="#00FF0D")

# ax[0].set_title('Speed Trace', pad=30)
ax[0].set_xlim(ref_dist[0],ref_dist[-1])
ax[0].set_ylim(0, 350)
ax[0].set_ylabel('Speed (km/h)')
ax[0].legend()
ax[0].grid(visible=False)

ax[1].plot(ref_lap['Distance'],ref_lap['Throttle'],color=ant_color,linewidth=3, label='2025 - VER')
ax[1].plot(comp_lap['Distance'],comp_lap['Throttle'],color=ham_color,linewidth=3, label='2026 - RUS')
# ax[1].vlines(x=corners['Distance'], ymin=-5, ymax=105,
#           linestyles='dotted', colors='grey')

ax[1].set_ylabel('Throttle (%)')
ax[1].set_ylim(-5, 105)
# ax[1].legend()
ax[1].grid(visible=False)

# ax[2].plot(ref_lap['Distance'],ref_lap['Brake'],color=ant_color,linewidth=3, label='2025 - VER')
# ax[2].plot(comp_lap['Distance'],comp_lap['Brake'],color=ham_color,linewidth=3, label='2026 - RUS')
# # ax[2].vlines(x=corners['Distance'], ymin=-0.1, ymax=1.1,
# #           linestyles='dotted', colors='grey')

# ax[2].set_ylabel('Brake (0 / 1)')
# ax[2].set_ylim(-0.1, 1.1)
# ax[2].set_xlabel('Lap Distance (m)')
# # ax[2].legend()
# ax[2].grid(visible=False)

fig.align_labels(axs=ax)

In [ ]:
save_fig(fig, 'Speed_Trace', 'Reel27', trs=True)